# 02. 모델 학습 및 실험

IF / LSTM / 앙상블 세 가지 비교  
threshold나 가중치 바꿔보면서 성능 튜닝하는 용도로 씀

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

from src.utils.data_utils import load_processed_data, train_val_split
from src.detection.detector import EnsembleDetector

plt.rcParams['figure.figsize'] = (12, 5)
RANDOM_STATE = 42

In [ ]:
agg_df, sequences, seq_labels = load_processed_data('../data/processed')

train_agg, val_agg, train_seq, val_seq, train_seq_labels, val_seq_labels = train_val_split(
    agg_df, sequences, seq_labels, val_ratio=0.2, random_state=RANDOM_STATE
)

print(f'Train: {len(train_agg)}명 / Val: {len(val_agg)}명')
print(f'Train 이상 비율: {train_agg["label"].mean():.2%}')
print(f'Val 이상 비율:   {val_agg["label"].mean():.2%}')

## 1. 앙상블 학습

In [ ]:
detector = EnsembleDetector(if_weight=0.4, lstm_weight=0.6, threshold=0.55)
detector.fit(train_agg, train_seq, train_seq_labels, lstm_epochs=30)

## 2. 검증 성능 평가

In [ ]:
val_labels = val_agg['label'].values
scores, preds = detector.evaluate(val_agg, val_seq, val_labels)

## 3. ROC Curve 비교

In [ ]:
if_scores, lstm_scores, ensemble_scores = detector.predict_scores(val_agg, val_seq)

fig, ax = plt.subplots(figsize=(8, 6))

for name, s in [('Isolation Forest', if_scores), ('LSTM AE', lstm_scores), ('앙상블', ensemble_scores)]:
    fpr, tpr, _ = roc_curve(val_labels, s)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve 비교')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료 → data/processed/roc_curve.png')

## 4. Threshold 튜닝
Precision-Recall 트레이드오프 확인

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.3, 0.9, 0.05)
results = []

for t in thresholds:
    preds_t = (ensemble_scores > t).astype(int)
    results.append({
        'threshold': round(t, 2),
        'precision': precision_score(val_labels, preds_t, zero_division=0),
        'recall': recall_score(val_labels, preds_t, zero_division=0),
        'f1': f1_score(val_labels, preds_t, zero_division=0),
        'flagged_rate': preds_t.mean(),
    })

results_df = pd.DataFrame(results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results_df['threshold'], results_df['precision'], label='Precision', marker='o')
ax1.plot(results_df['threshold'], results_df['recall'], label='Recall', marker='s')
ax1.plot(results_df['threshold'], results_df['f1'], label='F1', marker='^', linewidth=2)
ax1.axvline(x=0.55, color='gray', linestyle='--', alpha=0.6, label='현재 threshold')
ax1.set_xlabel('Threshold')
ax1.set_title('Precision / Recall / F1')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(results_df['threshold'], results_df['flagged_rate'], color='tomato', marker='o')
ax2.axvline(x=0.55, color='gray', linestyle='--', alpha=0.6)
ax2.set_xlabel('Threshold')
ax2.set_ylabel('플래그 비율')
ax2.set_title('Threshold별 플래그 비율')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nF1 최대 threshold:', results_df.loc[results_df['f1'].idxmax(), 'threshold'])
print(results_df.to_string(index=False))

## 5. 의심 플레이어 TOP 20

In [ ]:
report = detector.flag_report(val_agg, val_seq, top_n=20)